# Data Prep

In [1]:
import pandas as pd
import plotly.express as px
import unicodedata
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import time

/Users/victorzore/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Funcoes

In [2]:
def processar_movimentacoes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recebe o df com colunas mínimas:
      ['Data', 'EntradaSaida', 'Movimentacao', 'Produto',
       'Quantidade', 'Precounitario', 'ValordaOperacao']
    Retorna um novo DataFrame com colunas extras:
      ['Posicao', 'CustoTotal', 'CustoMedio'] após cada evento.
    """
    # 1) Preparação
    df = df.copy()
    df['Data'] = pd.to_datetime(df['Data'])
    df = df.sort_values(['Produto', 'Data']).reset_index(drop=True)
    # Quantidade com sinal: + para Crédito (compra), – para Débito (venda/resgate)
    df['dq'] = df['Quantidade'].where(df['EntradaSaida']=='Credito',
                                      -df['Quantidade'])
    
    # 2) estado inicial por produto
    estados = {}   # chave: produto → dict(qty, cost_total, avg_cost, frac_buffer)
    registros = [] # para montar o DataFrame de saída
    
    # 3) itera linha a linha
    for _, row in df.iterrows():
        p = row['Produto']
        if p not in estados:
            estados[p] = {'qty': 0.0,
                          'cost_total': 0.0,
                          'avg_cost': 0.0,
                          'frac_buffer': 0.0}
        st = estados[p]
        mov = row['Movimentacao']
        
        # ---- movimentações de mercado (compra/venda) ----
        if mov == 'Transferência - Liquidação':
            dq = row['dq']
            if dq > 0:
                # compra
                st['qty'] += dq
                st['cost_total'] += dq * row['Precounitario']
            else:
                # venda (resgate)
                sell_q = -dq
                st['cost_total'] -= sell_q * st['avg_cost']
                st['qty']       -= sell_q
            
        # ---- split / reverse split ----
        elif mov == 'Desdobro' or mov == 'Grupamento':
            # fator: nova_qty / antiga_qty = row['Quantidade'] / st['qty']
            fator = (row['Quantidade']  + st['qty'] )/ st['qty'] if st['qty']>0 else 1.0
            st['qty'] = st['qty'] * fator
            # custo total não muda → avg_cost cai em f¹¹
            # (recalcula avg_cost abaixo)
        
        # ---- bonificação gratuita ----
        elif mov == 'Bonificação em Ativos':
            # aqui row['Quantidade'] = nº de bônus
            fator = (st['qty'] + row['Quantidade']) / st['qty'] if st['qty']>0 else 1.0
            st['qty'] = st['qty'] + row['Quantidade']
            # cost_total inalterado
        
        # ---- fração / leilão de fração ----
        elif mov == 'Fração em Ativos':
            st['frac_buffer'] += row['Quantidade']
        elif mov == 'Leilão de Fração':
            qty = min(row['Quantidade'], st['frac_buffer'])
            price = row['ValordaOperacao'] / row['Quantidade']
            # registra entrada de caixa se precisar, mas ajusta buffer:
            st['frac_buffer'] -= qty
        
        # ---- resgate de fundos / units ----
        elif mov == 'Resgate':
            dq = -row['Quantidade']  # débito de posição
            st['qty']       -= row['Quantidade']
            st['cost_total'] -= row['Quantidade'] * st['avg_cost']
        
        # ---- atualização de evento anterior ----
        elif mov == 'Atualização':
            # você precisaria reprocessar o split/grupamento/bonificação
            # original com novo ratio – mais complexo, mas a ideia é 
            # sobrescrever o fator daquele evento e recalcular toda a 
            # série para frente.
            pass
        
        # 4) recalc avg cost se qty > 0
        st['avg_cost'] = st['cost_total'] / st['qty'] if st['qty']>0 else 0.0
        
        # 5) armazena snapshot pós-evento
        registros.append({
            'Data':         row['Data'],
            'Produto':      p,
            'Movimentacao': mov,
            'Quantidade':   row['Quantidade'],
            'dq':           row['dq'],
            'Posicao':      st['qty'],
            'CustoTotal':   st['cost_total'],
            'CustoMedio':   st['avg_cost'],
            'FracBuffer':   st['frac_buffer']
        })
    
    return pd.DataFrame(registros)

def get_google_price(ticker: str,
                     suffix: str = "BVMF",
                     pause: float = 1.0,
                     lang: str = "pt-BR") -> float:
    """
    Busca o preço atual de um ticker na página do Google Finance.

    :param ticker: Ex: 'PETR4', 'VALE3', 'XPML11'
    :param suffix: 'BVMF' para B3
    :param pause: segundos de espera entre requests
    :param lang: código de linguagem para cabeçalho 'Accept-Language'
    :return: preço como float
    """
    # adiciona timestamp para evitar cache
    url = f"https://www.google.com/finance/quote/{ticker}:{suffix}?nocache={int(time.time()*1000)}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept-Language": lang,
        "Cache-Control": "no-cache",
        "Pragma": "no-cache"
    }
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    html = r.text

    # 1) tenta extrair do JSON embutido, pegando o último match
    prices = re.findall(r'"regularMarketPrice"\s*:\s*\{\s*"raw"\s*:\s*([\d\.]+)', html)
    if prices:
        price = float(prices[-1])
        time.sleep(pause)
        return price

    # 2) fallback: scraping do elemento HTML
    soup = BeautifulSoup(html, "html.parser")
    el = soup.select_one("div.YMlKec.fxKbKc")
    if not el:
        raise ValueError(f"Preço não encontrado para {ticker}:{suffix}")
    txt = el.get_text().strip().replace("R$", "").replace(".", "").replace(",", ".")
    time.sleep(pause)
    return float(txt)

# Dados

In [4]:
# Faz a leitura da tabela
df21 = pd.read_excel('data-b3/mov21.xlsx')
df22 = pd.read_excel('data-b3/mov22.xlsx')
df23 = pd.read_excel('data-b3/mov23.xlsx')
df24 = pd.read_excel('data-b3/mov24.xlsx')
df25 = pd.read_excel('data-b3/mov25.xlsx')

# Concatena os dois dataframes
df_inicial = pd.concat([df21, df22, df23, df24, df25], ignore_index=True).drop_duplicates()

# Remove as transferencias
df = df_inicial[~(df_inicial['Movimentação'] == 'Transferência')]

# Transformando FBOK34 em M1TA34
df['Produto'] = df['Produto'].str.replace('FBOK34', 'M1TA34')

# Transformando a coluna Data no formato data
df['Data'] = pd.to_datetime(df_inicial['Data'], dayfirst=True, format='%d/%m/%Y')

# Trata os nomes das colunas: remove acentuação e caracteres especiais, mantendo letras, números e underscore
df.columns = (
    df.columns
      .str.strip()
      .map(lambda x: unicodedata.normalize('NFKD', x)
                            .encode('ASCII', errors='ignore')
                            .decode('ASCII'))
      .str.replace(r'[^A-Za-z0-9_]', '', regex=True)
)

# Transforma o valor do Produto em apenas o simbolo de cada produto
df['Produto'] = df['Produto'].astype(str).str.extract(r'^(\S+)', expand=False)

# Trasnforma em numerico as colunas que estao como texto.
df['Precounitario'] = pd.to_numeric(df['Precounitario'], errors='coerce')

/Users/victorzore/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/victorzore/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/victorzore/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/victorzore/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/victorzore/Library/Python

In [5]:
# Filtrar em uma nova variável apenas para excluir esse caso especial
mask_excluir = (
    (df['Movimentacao'] == 'Juros Sobre Capital Próprio - Transferido') &
    (df['EntradaSaida']      == 'Debito')
)
df = df.loc[~mask_excluir].copy()

# Substituir o texto na coluna Movimentacao usando .loc
mask_jscp = df['Movimentacao'] == 'Juros Sobre Capital Próprio - Transferido'
df.loc[mask_jscp, 'Movimentacao'] = 'Juros Sobre Capital Próprio'

# Filtrar rendimentos
rendimentos = [
    'Rendimento',
    'Dividendo',
    'Juros Sobre Capital Próprio'
]

df_rendimentos = df.loc[df['Movimentacao'].isin(rendimentos)].copy()

# Remove as linhas relacionadas a rendimentos
df = df.loc[~df['Movimentacao'].isin(rendimentos)].copy()

# Remove Renda Fixa
df = df.loc[~(df['Produto'] == 'CDB')].copy()

# Remover algumas movimentacoes:
remov_movimentacao = [
    'Direitos de Subscrição - Não Exercido',
    'Cessão de Direitos',
    'Cessão de Direitos - Solicitada',
    'Direito de Subscrição'
]

# Inverter o sinal de Precounitario quando for débito
df['Precounitario'] = np.where(
    df['EntradaSaida'] == 'Credito',
    df['Precounitario'],
    (-1)*df['Precounitario']
)

# Remove as movimentacoes que estao na lista remov_movimentacao
df = df.loc[~df['Movimentacao'].isin(remov_movimentacao)].copy()

In [6]:
# Uso:
df_processado = processar_movimentacoes(df)

# 1) Ordena só por garantia (Data já está no df_processado)
df_processado = df_processado.sort_values(['Produto','Data'])

# 2) Para cada Produto, pega a última linha
df_ultima_posicao = (
    df_processado
      .groupby('Produto', as_index=False)
      .last()   # mantém as colunas Data, Posicao, CustoTotal, CustoMedio, etc.
)

# 3) Se quiser, seleciona só as colunas-chave
df_ultima_posicao = df_ultima_posicao[[
    'Produto', 'Data', 'Posicao', 'CustoTotal', 'CustoMedio'
]]


##### # Antigo
# df_ativos = df[df['Movimentacao'] == 'Transferência - Liquidação'].copy()

# # Recalcular ValorDaOperacao como Quantidade * Precounitario
# df_ativos['ValordaOperacao'] = df_ativos['Quantidade'] * df_ativos['Precounitario']


# Rendimentos

In [7]:
# Supondo que df_rendimento já exista e contenha as colunas 'Produto' e 'Valor da Operação'
resumo_rendimentos = (
    df_rendimentos
    .groupby('Produto', as_index=False)['ValordaOperacao']
    .sum()
    .rename(columns={'ValordaOperacao': 'Total Rendimento'})
)

In [8]:
resumo_rendimentos['Total Rendimento'] = pd.to_numeric(resumo_rendimentos['Total Rendimento'], errors='coerce')

# Ordenar por valor
resumo_rendimentos = resumo_rendimentos.sort_values('Total Rendimento', ascending=True)

# Calcular valor total de rendimentos
valor_total = resumo_rendimentos['Total Rendimento'].sum()

# Criar gráfico de barras horizontal com tamanho ajustado
fig = px.bar(
    resumo_rendimentos,
    x='Total Rendimento',
    y='Produto',
    orientation='h',
    title=f'Total de Rendimento por Ação — Valor Total: R$ {valor_total:,.2f}',
    labels={'Total Rendimento': 'Valor Total (R$)', 'Produto': 'Ação'},
    height=800  # ajusta altura
)

fig.show()

# Preco Atual

In [ ]:
if __name__ == "__main__":
    # Supondo que df_ativos já exista e tenha a coluna 'Produto'
    
    # 1) Cria um DataFrame com valores únicos de 'Produto'
    valoratual = df_ultima_posicao[['Produto']].drop_duplicates().reset_index(drop=True)
    
    # 2) Lista para armazenar os preços
    precos = []
    
    # 3) Itera sobre cada produto e busca o preço
    for ativo in valoratual['Produto']:
        try:
            p = get_google_price(ativo)
        except Exception as e:
            print(f"Erro em {ativo}: {e}")
            p = 0.0
        print(f"O ativo vai ser buscado: {ativo} — Preço atual: R$ {p:.2f}")
        precos.append(p)
    
    # 4) Atribui a lista de preços como nova coluna
    valoratual['Preço_Atual'] = precos
    
    # 5) Exibe
    print(valoratual)

valoratual.to_csv('data-stocks/precosatuais.csv', index=False)

# Lendo a planilha de preços atuais
df_precos = pd.read_csv('data-stocks/precosatuais.csv')

O ativo vai ser buscado: AAPL34 — Preço atual: R$ 62.34
O ativo vai ser buscado: ABEV3 — Preço atual: R$ 12.12
O ativo vai ser buscado: AGRO3 — Preço atual: R$ 20.74
O ativo vai ser buscado: ALUP11 — Preço atual: R$ 29.44
O ativo vai ser buscado: AMER3 — Preço atual: R$ 5.54
O ativo vai ser buscado: AMZO34 — Preço atual: R$ 62.10
O ativo vai ser buscado: B3SA3 — Preço atual: R$ 12.52
O ativo vai ser buscado: BABA34 — Preço atual: R$ 23.65
O ativo vai ser buscado: BBAS3 — Preço atual: R$ 20.52
O ativo vai ser buscado: BBFI11 — Preço atual: R$ 348.48
O ativo vai ser buscado: BBSE3 — Preço atual: R$ 32.46
Erro em BCFF11: Preço não encontrado para BCFF11:BVMF
O ativo vai ser buscado: BCFF11 — Preço atual: R$ 0.00
O ativo vai ser buscado: BMOB3 — Preço atual: R$ 19.56
O ativo vai ser buscado: BPFF11 — Preço atual: R$ 54.52
O ativo vai ser buscado: BRCR11 — Preço atual: R$ 41.11
O ativo vai ser buscado: BRKM3 — Preço atual: R$ 9.48
O ativo vai ser buscado: BRSR6 — Preço atual: R$ 11.35
O ati

In [28]:
df_precos

,Produto,Preço_Atual
0,AAPL34,62.34
1,ABEV3,12.12
2,AGRO3,20.74
3,ALUP11,29.44
4,AMER3,5.54
...,...,...
81,XPBR31,91.35
82,XPLG11,98.10
83,XPML11,99.24
84,XPSF11,5.99


# Carteira

In [26]:
df_atual = df_ultima_posicao.merge(df_precos, on='Produto', how='left')
carteira = df_atual[df_atual['Posicao'] > 0].copy()
carteira = carteira[carteira['Preço_Atual'] > 0].copy()

# Rotular o tipo de ativo 
carteira['Tipo'] = np.where(
    carteira['Produto'].str.contains('11'),
    'Fundo Imobiliário',
    'Ação'
)

carteira['TotalAtual'] = carteira['Posicao'] * carteira['Preço_Atual']
carteira['Valorpago'] = carteira['CustoMedio'] * carteira['Posicao']
carteira['LucroPrejuizo'] = carteira['TotalAtual'] - carteira['Valorpago']
carteira['LPPercentual'] = (carteira['LucroPrejuizo'] / carteira['Valorpago']) * 100


# Join dos rendimentos com a carteira
carteira = carteira.merge(resumo_rendimentos, on='Produto', how='left')
carteira['Total Rendimento'] = carteira['Total Rendimento'].fillna(0.0)
carteira['LPTotal'] = carteira['TotalAtual'] + carteira['Total Rendimento']
carteira['LPTotalliq'] = carteira['LPTotal'] - carteira['Valorpago']
carteira['LPTotalPercentual'] = (carteira['LPTotalliq'] / carteira['Valorpago']) * 100

In [27]:
# Prints com a situacao da carteira
print(f"Valor total:",  sum(carteira['TotalAtual']))
print(f"Valor pago: ", sum(carteira['Valorpago']))
print(f"Lucro/Prejuizo: ", sum(carteira['LucroPrejuizo']))
print(f"Percentual: ", np.mean(carteira['LPPercentual']))
print(f"Lucro/Prejuizo Total: ", sum(carteira['LPTotal']))
print(f"Percentual Total: ", np.mean(carteira['LPTotalPercentual']))

Valor total: 124007.304
Valor pago:  122950.92721882311
Lucro/Prejuizo:  1056.3767811769153
Percentual:  1.1199353764376263
Lucro/Prejuizo Total:  131524.074
Percentual Total:  6.24844641807519


In [12]:
import plotly.express as px

fig = px.scatter(
    carteira,
    x='Posicao',
    y='LPPercentual',
    hover_name='Produto',
    color='Tipo',
    size='Total Rendimento',
    title='Lucro/Prejuízo (%) vs Posição',
    labels={
        'Produto': 'Produto',
        'Posicao': 'Posição',
        'LPPercentual': 'Lucro/Prejuízo (%)',
        'Total Rendimento': 'Total Rendimento',
        'Tipo': 'Tipo'
    }
)

fig.show()


# Desdobramento

In [14]:
# def add_split_factor(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Adiciona coluna 'fator' ao DataFrame indicando o fator de desdobro 
#     para cada linha onde Movimentacao == 'Desdobro'. Para outras linhas, fica NaN.

#     Parâmetros:
#     - df: DataFrame com colunas ['Data', 'Movimentacao', 'Quantidade', 'Produto']

#     Retorna:
#     - DataFrame com a coluna 'fator' adicionada.
#     """
#     # Copia e prepara
#     df = df.copy()
#     df['Data'] = pd.to_datetime(df['Data'])
#     df = df.sort_values(['Produto', 'Data']).reset_index(drop=True)
#     df['fator'] = np.nan

#     # Itera por produto
#     for produto, grupo in df.groupby('Produto', sort=False):
#         # índices relativos ao df original
#         split_idxs = grupo.index[grupo['Movimentacao'] == 'Desdobro']
#         for idx in split_idxs:
#             split_date = df.at[idx, 'Data']
#             split_qty  = df.at[idx, 'Quantidade']
#             # soma quantidades anteriores do mesmo produto
#             existing_qty = df.loc[
#                 (df['Produto'] == produto) & (df['Data'] < split_date),
#                 'Quantidade'
#             ].sum()
#             if existing_qty > 0:
#                 df.at[idx, 'fator'] = (split_qty +  existing_qty) / existing_qty
#             # caso contrário, permanece NaN
    
#     df['fator'] = df['fator']

#     return df


In [15]:
# # df = seu DataFrame com colunas Data, Movimentacao, Quantidade, Produto
# df_com_fator = add_split_factor(df)

In [13]:
def adjust_for_splits(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Ajusta o DataFrame para splits:
      1) Encontra cada evento de 'Desdobro' e calcula o fator.
      2) Divide os valores de 'Precounitario' anteriores ao split pelo fator.
      3) Salva cada evento de split num DataFrame df_desdobro (Data, Produto, fator).
      4) Remove as linhas de 'Desdobro' do DataFrame original.
    
    Parâmetros:
    - df: DataFrame com colunas ['Data', 'Movimentacao', 'Produto',
                                  'Quantidade', 'Precounitario', ...]
    
    Retorna:
    - df_adj: DataFrame ajustado (sem linhas de Desdobro, com Precounitario corrigido)
    - df_desdobro: DataFrame com ['Data', 'Produto', 'fator'] para cada split
    """
    # 1) Prepara o DataFrame
    df = df.copy()
    df['Data'] = pd.to_datetime(df['Data'])
    df = df.sort_values(['Produto', 'Data']).reset_index(drop=True)
    df['fator'] = np.nan
    
    # 2) Itera por produto e identifica splits
    records = []
    for produto, grupo in df.groupby('Produto', sort=False):
        split_idxs = grupo.index[grupo['Movimentacao'] == 'Desdobro']
        for idx in split_idxs:
            split_date = df.at[idx, 'Data']
            split_qty  = df.at[idx, 'Quantidade']
            # total de ações antes do split
            existing_qty = df.loc[
                (df['Produto'] == produto) & (df['Data'] < split_date),
                'Quantidade'
            ].sum()
            
            if existing_qty > 0:
                factor = (split_qty + existing_qty) / existing_qty
                # registra o fator na linha de split
                df.at[idx, 'fator'] = factor
                
                # 3) Ajusta Precounitario das linhas anteriores
                mask = (df['Produto'] == produto) & (df['Data'] < split_date)
                df.loc[mask, 'Precounitario'] = df.loc[mask, 'Precounitario'] / factor
                df.loc[mask, 'Quantidade'] = df.loc[mask, 'Quantidade'] * factor
                
                # 4) Armazena o evento de split
                records.append({
                    'Data':    split_date,
                    'Produto': produto,
                    'fator':   factor
                })
    
    # 5) Constrói df_desdobro e remove linhas de split do df ajustado
    df_desdobro = pd.DataFrame(records, columns=['Data', 'Produto', 'fator'])
    df_adj = df[df['Movimentacao'] != 'Desdobro'].drop(columns=['fator']).reset_index(drop=True)
    
    return df_adj, df_desdobro


In [14]:
# Exemplo de uso:
df_adj, df_desdobro = adjust_for_splits(df)

In [25]:
df_adj

,EntradaSaida,Data,Movimentacao,Produto,Instituicao,Quantidade,Precounitario,ValordaOperacao,Resultado
0,Credito,2021-02-10,Transferência - Liquidação,AAPL34,C6 CORRETORA DE TITULOS E VALORES MOBILIARIOS ...,5.0,72.65,363.25,363.25
1,Debito,2021-03-24,Transferência - Liquidação,AAPL34,C6 CORRETORA DE TITULOS E VALORES MOBILIARIOS ...,5.0,-67.92,339.6,-339.60
2,Credito,2023-01-23,Transferência - Liquidação,AAPL34,NU INVEST CORRETORA DE VALORES S.A.,4.0,35.06,140.24,140.24
3,Credito,2023-01-30,Transferência - Liquidação,AAPL34,NU INVEST CORRETORA DE VALORES S.A.,3.0,36.54,109.62,109.62
4,Credito,2023-02-06,Transferência - Liquidação,AAPL34,NU INVEST CORRETORA DE VALORES S.A.,3.0,37.17,111.51,111.51
...,...,...,...,...,...,...,...,...,...
836,Credito,2023-07-25,Transferência - Liquidação,XPSF11,NU INVEST CORRETORA DE VALORES S.A.,8.0,8.30,66.4,66.40
837,Credito,2023-08-18,Transferência - Liquidação,XPSF11,NU INVEST CORRETORA DE VALORES S.A.,2.0,8.66,17.32,17.32
838,Credito,2023-10-18,Transferência - Liquidação,XPSF11,NU INVEST CORRETORA DE VALORES S.A.,3.0,8.29,24.87,24.87
839,Debito,2023-10-24,Transferência - Liquidação,XPSF11,NU INVEST CORRETORA DE VALORES S.A.,121.0,-8.09,978.89,-978.89


In [21]:
df_adj['Resultado'] = df_adj['Precounitario']*df_adj['Quantidade']

In [24]:
sum(df_adj['Resultado'].dropna())

117503.58269999997